# 04 - Analise Estatistica da Segmentacao Bruta

Carrega a base analitica da segmentacao bruta a partir do SQLite, persiste os resultados estatisticos no banco e exibe uma leitura inicial para comparacao entre modelos e dificuldades.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import (
    MetricsCollector,
    build_and_persist_analysis_segmentacao_bruta_estabilidade,
    build_and_persist_analysis_segmentacao_bruta_interacao_tag_modelo,
    build_and_persist_analysis_segmentacao_bruta_intervalo_confianca,
    build_and_persist_analysis_segmentacao_bruta_resumo_execucao,
    build_and_persist_analysis_segmentacao_bruta_resumo_modelo,
    build_and_persist_analysis_segmentacao_bruta_resumo_tag,
    build_and_persist_analysis_segmentacao_bruta_testes_modelo,
    build_and_persist_analysis_segmentacao_bruta_testes_tag,
)
from src.repositories import (
    AnaliseSegmentacaoBrutaEstabilidadeRepository,
    AnaliseSegmentacaoBrutaInteracaoTagModeloRepository,
    AnaliseSegmentacaoBrutaIntervaloConfiancaRepository,
    AnaliseSegmentacaoBrutaResumoExecucaoRepository,
    AnaliseSegmentacaoBrutaResumoModeloRepository,
    AnaliseSegmentacaoBrutaResumoTagRepository,
    AnaliseSegmentacaoBrutaTesteModeloRepository,
    AnaliseSegmentacaoBrutaTesteTagRepository,
)


## Carregamento da base analitica em memoria

Le a avaliacao persistida no SQLite e monta a base analitica por `imagem + modelo + execucao` em memoria.


In [ ]:
collector = MetricsCollector(force_recalculate=False)
df_base = collector.collect_all_metrics()

print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes: {df_base["execucao"].nunique()}')
print(f'Tags binarias disponiveis: {len([column for column in df_base.columns if column.startswith("tag_")])}')

df_base.head()


## Persistencia da camada analitica

Calcula os resumos descritivos, estabilidade, intervalos de confianca, comparacoes entre modelos, testes por tag e interacoes `modelo x tag`, persistindo tudo no SQLite.


In [ ]:
resumo_modelo_repository = AnaliseSegmentacaoBrutaResumoModeloRepository()
resumo_execucao_repository = AnaliseSegmentacaoBrutaResumoExecucaoRepository()
resumo_tag_repository = AnaliseSegmentacaoBrutaResumoTagRepository()
estabilidade_repository = AnaliseSegmentacaoBrutaEstabilidadeRepository()
intervalo_confianca_repository = AnaliseSegmentacaoBrutaIntervaloConfiancaRepository()
teste_modelo_repository = AnaliseSegmentacaoBrutaTesteModeloRepository()
teste_tag_repository = AnaliseSegmentacaoBrutaTesteTagRepository()
interacao_tag_modelo_repository = AnaliseSegmentacaoBrutaInteracaoTagModeloRepository()

df_resumo_modelo, _ = build_and_persist_analysis_segmentacao_bruta_resumo_modelo(
    df_base=df_base,
    repository=resumo_modelo_repository,
)
df_resumo_execucao, _ = build_and_persist_analysis_segmentacao_bruta_resumo_execucao(
    df_base=df_base,
    repository=resumo_execucao_repository,
)
df_resumo_tag, _ = build_and_persist_analysis_segmentacao_bruta_resumo_tag(
    df_base=df_base,
    repository=resumo_tag_repository,
)
df_estabilidade, _ = build_and_persist_analysis_segmentacao_bruta_estabilidade(
    df_base=df_base,
    repository=estabilidade_repository,
)
df_intervalo_confianca, _ = build_and_persist_analysis_segmentacao_bruta_intervalo_confianca(
    df_base=df_base,
    repository=intervalo_confianca_repository,
)
df_testes_modelo, _ = build_and_persist_analysis_segmentacao_bruta_testes_modelo(
    df_base=df_base,
    repository=teste_modelo_repository,
)
df_testes_tag, _ = build_and_persist_analysis_segmentacao_bruta_testes_tag(
    df_base=df_base,
    repository=teste_tag_repository,
)
df_interacoes_tag_modelo, _ = build_and_persist_analysis_segmentacao_bruta_interacao_tag_modelo(
    df_base=df_base,
    repository=interacao_tag_modelo_repository,
)

print(f'Registros no resumo por modelo: {len(df_resumo_modelo)}')
print(f'Registros no resumo por execucao: {len(df_resumo_execucao)}')
print(f'Registros no resumo por tag: {len(df_resumo_tag)}')
print(f'Registros na estabilidade: {len(df_estabilidade)}')
print(f'Registros nos intervalos de confianca: {len(df_intervalo_confianca)}')
print(f'Registros nos testes entre modelos: {len(df_testes_modelo)}')
print(f'Registros nos testes por tag: {len(df_testes_tag)}')
print(f'Registros nas interacoes modelo x tag: {len(df_interacoes_tag_modelo)}')


## Leitura inicial dos resultados

Mostra uma visao compacta das tabelas persistidas para orientar a etapa visual do notebook 05.


In [ ]:
pd.pivot_table(
    df_resumo_modelo,
    index='modelo',
    columns='metric_name',
    values=['mean', 'median'],
)


In [ ]:
df_estabilidade.sort_values(['metric_name', 'cv_execucoes', 'amplitude_execucoes'])


In [ ]:
df_intervalo_confianca.sort_values(['metric_name', 'statistic_name', 'modelo']).head(12)


In [ ]:
if df_testes_modelo.empty:
    print('Nao ha modelos suficientes no banco atual para executar comparacoes entre modelos.')
else:
    df_testes_modelo.sort_values(['metric_name', 'comparison_scope', 'p_value_adjusted'])


In [ ]:
df_testes_tag.sort_values(['metric_name', 'tag_name', 'comparison_scope', 'p_value_adjusted']).head(20)


In [ ]:
df_interacoes_tag_modelo.sort_values(['metric_name', 'tag_name', 'adjusted_delta_mean']).head(20)
